# Music Annotation Project 59R4A
## Targeted Adjudication of Page/Scan Edges, Region Fragments and Model Disagreements

**Runtime:** CPU. Run after Notebook 59R4.

This notebook does not retrain a model. It preserves the original Notebook 66 decisions and creates a separate `review_adjudications.csv` containing only explicit later corrections.

The review queue is deliberately small and targeted:
- all human `UNCERTAIN` cases;
- apparent page/scan-edge candidates;
- multiscale regressions against the scalar/local33 arms;
- high-disagreement cases;
- visibly incomplete region fragments.

The reviewer records one required outcome:

`ANNOTATION / PRINT / MIXED / PAGE_SCAN_EDGE / UNCERTAIN / REGION_FRAGMENT / MASK_ERROR`

`REGION_FRAGMENT` is an exclusion/status for a semantically incomplete candidate region, not a correction class.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q gradio pandas numpy opencv-python-headless pillow


Mounted at /content/drive


In [7]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, zipfile
import numpy as np
import pandas as pd
import cv2
import gradio as gr
from PIL import Image

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
R59R4_ROOT = ROOT / 'notebook59R4_multiscale_region_semantics'
OUTPUT_ROOT = ROOT / 'region_semantic_adjudication_59R4A'

SOURCE_59R4_RUN_OVERRIDE = None
DECISIONS_CSV_OVERRIDE = None
MAX_QUEUE = 60
PUBLIC_LINK = True
BORDER_MARGIN_FRACTION = 0.025
DISAGREEMENT_MODELS = ['pred_scalar','pred_local33','pred_multiscale33_97_257']

RUN = datetime.now().strftime('%Y%m%d_%H%M%S') + '_59R4A_targeted_adjudication'
OUT = OUTPUT_ROOT / RUN
TABLES = OUT/'tables'; REVIEW = OUT/'review'; AUDIT = OUT/'audit'; UPLOAD = OUT/'upload'
for p in [OUT,TABLES,REVIEW,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)

def require(ok,msg):
    if not ok: raise RuntimeError(msg)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def atomic_json(path,value):
    tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)

def complete_run(path):
    return all((path/x).exists() for x in [
        'tables/out_of_fold_predictions.csv','tables/model_summary.csv','audit/notebook59R4_verdict.json'
    ])

def latest_complete(root):
    hits=[]
    if root.exists():
        for p in root.rglob('*'):
            if p.is_dir() and complete_run(p): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def find_unique_below(root,name,label,required=True):
    hits=sorted({p.resolve() for p in Path(root).rglob(name) if p.is_file()})
    if not hits and not required: return None
    require(len(hits)==1,f'{label}: expected one {name} below {root}, found {len(hits)}: {hits[:20]}')
    return hits[0]

SOURCE = Path(SOURCE_59R4_RUN_OVERRIDE) if SOURCE_59R4_RUN_OVERRIDE else latest_complete(R59R4_ROOT)
require(SOURCE is not None and complete_run(SOURCE),'No complete Notebook 59R4 run found. Set SOURCE_59R4_RUN_OVERRIDE.')
PRED_PATH = SOURCE/'tables'/'out_of_fold_predictions.csv'
DECISIONS_PATH = Path(DECISIONS_CSV_OVERRIDE) if DECISIONS_CSV_OVERRIDE else find_unique_below(SOURCE,'notebook66_completed_snapshot.csv','Notebook 66 decisions')
print('59R4:',SOURCE)
print('Decisions:',DECISIONS_PATH)
print('Output:',OUT)


59R4: /content/drive/MyDrive/MusicAnnotationProject/notebook59R4_multiscale_region_semantics/20261114_165044_notebook59R4
Decisions: /content/drive/MyDrive/MusicAnnotationProject/notebook59R4_multiscale_region_semantics/20261114_165044_notebook59R4/inputs/notebook66_completed_snapshot.csv
Output: /content/drive/MyDrive/MusicAnnotationProject/region_semantic_adjudication_59R4A/20261115_091326_59R4A_targeted_adjudication


In [ ]:
pred=pd.read_csv(PRED_PATH,dtype=str).fillna('')
dec=pd.read_csv(DECISIONS_PATH,dtype=str).fillna('')
required={'region_id','page','semantic_label','annotation_family'}
require(required.issubset(dec.columns),f'review_decisions.csv missing {sorted(required-set(dec.columns))}')
require('region_id' in pred.columns,'59R4 predictions lack region_id')

# Preserve the original review as immutable evidence.
shutil.copy2(DECISIONS_PATH,AUDIT/'review_decisions_original.csv')
shutil.copy2(PRED_PATH,AUDIT/'out_of_fold_predictions_original.csv')

work=dec.merge(pred,on='region_id',how='left',validate='one_to_one',suffixes=('','_59R4'))
require(work['page'].ne('').all(),'Missing page identity after merge')

# ------------------------------------------------------------
# Resolve the actual Notebook 59R4 prediction columns
# ------------------------------------------------------------

def first_col(
    frame,
    names,
):
    return next(
        (
            name
            for name in names
            if name in frame.columns
        ),
        None,
    )


# First print the available prediction-like columns.
prediction_like_columns = [
    column
    for column in work.columns
    if (
        'pred' in column.lower()
        or 'scalar' in column.lower()
        or 'local33' in column.lower()
        or 'multiscale' in column.lower()
        or 'multi' in column.lower()
    )
]


print(
    'Prediction-like columns found in merged 59R4 data:'
)

for column in prediction_like_columns:

    print(
        '  ',
        repr(
            column
        ),
    )


# Resolve using both the intended 59R4 names and sensible
# aliases produced by earlier versions of the notebook.

model_cols = {

    'scalar':
        first_col(
            work,
            [
                'pred_scalar',
                'prediction_scalar',
                'scalar_prediction',
                'scalar_pred',
                'predicted_scalar',
                'semantic_pred_scalar',
                'semantic_prediction_scalar',
            ],
        ),

    'local33':
        first_col(
            work,
            [
                'pred_local33',
                'prediction_local33',
                'local33_prediction',
                'local33_pred',
                'predicted_local33',
                'semantic_pred_local33',
                'semantic_prediction_local33',
            ],
        ),

    'multi':
        first_col(
            work,
            [
                'pred_multiscale33_97_257',
                'pred_multiscale_33_97_257',
                'prediction_multiscale33_97_257',
                'prediction_multiscale_33_97_257',
                'multiscale33_97_257_prediction',
                'multiscale_33_97_257_prediction',
                'multiscale_prediction',
                'multiscale_pred',
                'pred_multiscale',
                'pred_multi',
                'multi_prediction',
                'multi_pred',
                'semantic_pred_multiscale33_97_257',
                'semantic_pred_multiscale_33_97_257',
            ],
        ),
}


print(
    '\nResolved 59R4 model columns:'
)

for model_name, column_name in model_cols.items():

    print(
        f'  {model_name:9s} -> {column_name}'
    )


unresolved_models = [
    model_name
    for model_name, column_name
    in model_cols.items()
    if column_name is None
]


require(
    not unresolved_models,
    (
        'Could not resolve these Notebook 59R4 '
        'prediction columns: '
        f'{unresolved_models}. '
        'See the prediction-like column list printed above.'
    ),
)


# Create stable internal names used by the rest of 59R4A.

for model_name, column_name in model_cols.items():

    work[
        f'_pred_{model_name}'
    ] = (
        work[
            column_name
        ]
        .astype(str)
        .str.upper()
        .str.strip()
    )


print(
    '\nResolved prediction value counts:'
)


for model_name in [
    'scalar',
    'local33',
    'multi',
]:

    column = (
        f'_pred_{model_name}'
    )

    print(
        '\n',
        model_name,
    )

    print(
        work[
            column
        ].value_counts(
            dropna=False
        )
    )

allowed_prediction_labels = {
    'ANNOTATION',
    'PRINT',
    'MIXED',
    'UNCERTAIN',
    'PAGE_SCAN_EDGE',
}


for model_name in [
    'scalar',
    'local33',
    'multi',
]:

    column = (
         f'_pred_{model_name}'
    )

    observed_labels = set(
        work[
            column
        ].dropna()
    )

    unexpected = (
        observed_labels
        - allowed_prediction_labels
    )

    require(
        not unexpected,
        (
            f'{model_name}: unexpected prediction labels '
            f'in {column}: {sorted(unexpected)}'
        ),
    )

# Geometry aliases used only for queue prioritisation, never as labels.
def num(names,default=np.nan):
    c=first_col(work,names)
    return pd.to_numeric(work[c],errors='coerce') if c else pd.Series(default,index=work.index,dtype=float)
work['_x0']=num(['x0','bbox_x0','x','left'])
work['_y0']=num(['y0','bbox_y0','y','top'])
work['_x1']=num(['x1','bbox_x1','right'])
work['_y1']=num(['y1','bbox_y1','bottom'])
work['_page_w']=num(['page_width','image_width','page_w'])
work['_page_h']=num(['page_height','image_height','page_h'])
if work['_x1'].isna().all(): work['_x1']=work['_x0']+num(['bbox_w','w','width'])
if work['_y1'].isna().all(): work['_y1']=work['_y0']+num(['bbox_h','h','height'])

border=(
 (work['_x0']<=BORDER_MARGIN_FRACTION*work['_page_w']) |
 (work['_y0']<=BORDER_MARGIN_FRACTION*work['_page_h']) |
 (work['_x1']>=(1-BORDER_MARGIN_FRACTION)*work['_page_w']) |
 (work['_y1']>=(1-BORDER_MARGIN_FRACTION)*work['_page_h'])
).fillna(False)
uncertain=work.semantic_label.str.upper().eq('UNCERTAIN')
scalar_reg=(work['_pred_scalar'].eq(work.semantic_label.str.upper()) & ~work['_pred_multi'].eq(work.semantic_label.str.upper()))
local_reg=(work['_pred_local33'].eq(work.semantic_label.str.upper()) & ~work['_pred_multi'].eq(work.semantic_label.str.upper()))
disagree=(work[['_pred_scalar','_pred_local33','_pred_multi']].nunique(axis=1)>1)

work['_queue_reason']=''
for mask,label in [(uncertain,'HUMAN_UNCERTAIN'),(border,'BORDER_CANDIDATE'),(scalar_reg,'MULTISCALE_REGRESSION_VS_SCALAR'),(local_reg,'MULTISCALE_REGRESSION_VS_LOCAL33'),(disagree,'MODEL_DISAGREEMENT')]:
    work.loc[mask,'_queue_reason']=work.loc[mask,'_queue_reason'].map(lambda x:'|'.join(filter(None,[x,label])))
queue=work[work['_queue_reason'].ne('')].copy()
queue['_priority']=uncertain.loc[queue.index].astype(int)*100 + border.loc[queue.index].astype(int)*50 + scalar_reg.loc[queue.index].astype(int)*20 + local_reg.loc[queue.index].astype(int)*10 + disagree.loc[queue.index].astype(int)*5
queue=queue.sort_values(['_priority','page'],ascending=[False,True]).drop_duplicates('region_id')
if MAX_QUEUE is not None: queue=queue.head(MAX_QUEUE)
require(len(queue)>0,'Targeted 59R4A queue is empty.')
queue['review_order']=np.arange(1,len(queue)+1)
queue.to_csv(TABLES/'adjudication_queue.csv',index=False)
print('Queue:',len(queue),'regions across',queue.page.nunique(),'pages')
print(queue._queue_reason.value_counts().head(20))


RuntimeError: Could not resolve model prediction columns: {'scalar': 'scalar_prediction', 'local33': 'local33_prediction', 'multi': None}

In [ ]:
# ============================================================
# Resolve authoritative observed-page images for the
# adjudication queue.
#
# Do NOT assume the source images were copied into the 59R4
# run directory.
#
# Resolution priority:
#
#   1. explicit image path already stored in the row;
#   2. Notebook 64 / 61 style observed page path in the row;
#   3. exact page stem under ROOT/aligned;
#   4. exact page stem under ROOT/source/annotated_only;
#   5. Notebook 07 review audit aligned_output;
#   6. downstream observed.png artefact only as a final
#      fallback.
#
# Ambiguous matches fail closed.
# ============================================================

SUPPORTED_IMAGES = {
    '.png',
    '.jpg',
    '.jpeg',
    '.tif',
    '.tiff',
    '.bmp',
}


# ------------------------------------------------------------
# 1. Basic helpers
# ------------------------------------------------------------

def image_map(
    folder,
):

    folder = Path(
        folder
    )

    if not folder.exists():

        return {}


    result = {}


    for path in folder.iterdir():

        if (
            path.is_file()
            and path.suffix.lower()
            in SUPPORTED_IMAGES
        ):

            # Do not silently overwrite duplicate stems.

            stem = path.stem

            if stem in result:

                raise RuntimeError(
                    'Duplicate image stem in '
                    f'{folder}: {stem}'
                )

            result[
                stem
            ] = path


    return result


def existing_path(
    value,
):

    if value is None:

        return None


    text = str(
        value
    ).strip()


    if (
        not text
        or text.lower()
        in {
            'nan',
            'none',
        }
    ):

        return None


    path = Path(
        text
    )


    if path.is_file():

        return path


    # Relative path beneath project root.

    candidate = (
        ROOT
        / path
    )

    if candidate.is_file():

        return candidate


    # Relative path beneath 59R4 run.

    candidate = (
        SOURCE
        / path
    )

    if candidate.is_file():

        return candidate


    return None


# ------------------------------------------------------------
# 2. Authoritative project image maps
# ------------------------------------------------------------

ALIGNED_MAP = image_map(
    ROOT
    / 'aligned'
)


ANNOTATED_ONLY_MAP = image_map(
    ROOT
    / 'source'
    / 'annotated_only'
)


print(
    'ROOT/aligned pages:',
    len(
        ALIGNED_MAP
    )
)

print(
    'ROOT/source/annotated_only pages:',
    len(
        ANNOTATED_ONLY_MAP
    )
)


# ------------------------------------------------------------
# 3. Load Notebook 07 review audits
#
# Notebook 09 establishes this as the authoritative method for
# resolving a reviewed annotated-only page: the review audit
# records aligned_output and corrected_mask_output.
# ------------------------------------------------------------

REVIEW_AUDIT_DIR = (
    ROOT
    / 'human_in_the_middle'
    / 'audit'
)


review_audits = []


if REVIEW_AUDIT_DIR.exists():

    for path in sorted(
        REVIEW_AUDIT_DIR.glob(
            '*.json'
        )
    ):

        try:

            value = json.loads(
                path.read_text()
            )

            if not isinstance(
                value,
                dict,
            ):

                continue


            value[
                '_audit_path'
            ] = str(
                path
            )


            review_audits.append(
                value
            )


        except Exception as error:

            raise RuntimeError(
                'Unreadable Notebook 07 review audit: '
                f'{path}: {error}'
            )


print(
    'Notebook 07 review audits:',
    len(
        review_audits
    )
)


# ------------------------------------------------------------
# 4. Build several audit lookup identities
#
# Different Notebook 07 revisions recorded slightly different
# path/name fields. Use all safe exact identities available.
# ------------------------------------------------------------

AUDIT_PAGE_LOOKUP = {}


def add_audit_page_identity(
    page,
    record,
):

    if page is None:

        return


    page = str(
        page
    ).strip()


    if not page:

        return


    # Normalise a filename/path to its stem where appropriate.

    try:

        page = Path(
            page
        ).stem

    except Exception:

        pass


    if page not in AUDIT_PAGE_LOOKUP:

        AUDIT_PAGE_LOOKUP[
            page
        ] = []


    if record not in AUDIT_PAGE_LOOKUP[
        page
    ]:

        AUDIT_PAGE_LOOKUP[
            page
        ].append(
            record
        )


for record in review_audits:

    # Direct page-like fields.

    for key in [
        'page',
        'page_id',
        'source_stem',
        'stem',
        'name',
    ]:

        if record.get(
            key
        ):

            add_audit_page_identity(
                record[
                    key
                ],
                record,
            )


    # Source/path fields can also identify Page258 etc.

    for key in [
        'source',
        'source_file',
        'aligned_output',
        'corrected_mask_output',
    ]:

        raw = record.get(
            key
        )

        if raw:

            add_audit_page_identity(
                raw,
                record,
            )


# ------------------------------------------------------------
# 5. Explicit row-path fields
#
# These vary slightly by Notebook 59R3/59R4 revision.
# ------------------------------------------------------------

ROW_IMAGE_COLUMNS = [
    'observed_path',
    'observed_image_path',
    'page_path',
    'page_image_path',
    'aligned_path',
    'aligned_output',
    'source_path',
    'source_file',
    'image_path',
    'context_257_path',
    'context_path',
    'review_image_path',
    'crop_path',
]


# ------------------------------------------------------------
# 6. Resolve a Notebook 07 audit record by exact page stem
# ------------------------------------------------------------

def resolve_from_review_audit(
    page,
):

    records = (
        AUDIT_PAGE_LOOKUP.get(
            page,
            []
        )
    )


    resolved = []


    for record in records:

        raw = record.get(
            'aligned_output'
        )


        path = existing_path(
            raw
        )


        if path is not None:

            resolved.append(
                path
            )


    # Deduplicate exact paths.

    resolved = sorted(
        {
            path.resolve()
            for path
            in resolved
        }
    )


    if len(
        resolved
    ) > 1:

        raise RuntimeError(
            f'{page}: multiple Notebook 07 '
            f'aligned_output files resolve: {resolved}'
        )


    return (
        resolved[
            0
        ]
        if resolved
        else None
    )


# ------------------------------------------------------------
# 7. Main resolver
# ------------------------------------------------------------

def resolve_image(
    row,
):

    page = str(
        row[
            'page'
        ]
    ).strip()


    # --------------------------------------------------------
    # A. Explicit durable path already stored in row
    # --------------------------------------------------------

    for column in ROW_IMAGE_COLUMNS:

        if (
            column
            in row.index
        ):

            path = existing_path(
                row[
                    column
                ]
            )


            if path is not None:

                return (
                    path,
                    f'row:{column}'
                )


    # --------------------------------------------------------
    # B. Authoritative aligned page
    # --------------------------------------------------------

    if page in ALIGNED_MAP:

        return (
            ALIGNED_MAP[
                page
            ],
            'ROOT/aligned',
        )


    # --------------------------------------------------------
    # C. Annotated-only source page
    #
    # Use only if no aligned reviewed page was found.
    # --------------------------------------------------------

    if page in ANNOTATED_ONLY_MAP:

        return (
            ANNOTATED_ONLY_MAP[
                page
            ],
            'ROOT/source/annotated_only',
        )


    # --------------------------------------------------------
    # D. Notebook 07 review audit
    # --------------------------------------------------------

    audit_path = (
        resolve_from_review_audit(
            page
        )
    )


    if audit_path is not None:

        return (
            audit_path,
            'Notebook07:aligned_output',
        )


    # --------------------------------------------------------
    # E. Final fallback: search only for an exact observed.png
    # under known experiment runs where the parent directory
    # itself is the page name.
    #
    # Do NOT recursively accept arbitrary PNGs named after the
    # region ID.
    # --------------------------------------------------------

    fallback_roots = [
        ROOT
        / '59r2_annotated_only_shadow_test',

        ROOT
        / 'region_semantic_challenger_59R4',

        SOURCE,
    ]


    candidates = []


    for root in fallback_roots:

        if not root.exists():

            continue


        for path in root.rglob(
            'observed.png'
        ):

            if (
                path.parent.name
                == page
            ):

                candidates.append(
                    path.resolve()
                )


    candidates = sorted(
        set(
            candidates
        )
    )


    if len(
        candidates
    ) > 1:

        # Multiple historical observed copies may legitimately
        # exist. Require byte identity before choosing one.

        hashes = {
            sha256_file(
                path
            )
            for path
            in candidates
        }


        if len(
            hashes
        ) != 1:

            raise RuntimeError(
                f'{page}: multiple non-identical '
                'observed-page fallbacks were found: '
                f'{candidates}'
            )


    if candidates:

        return (
            candidates[
                -1
            ],
            'historical_observed_copy',
        )


    return (
        None,
        None,
    )


# ------------------------------------------------------------
# 8. Resolve every queue row
# ------------------------------------------------------------

resolved_paths = []

resolved_sources = []


for _,
row in queue.iterrows():

    path, source = (
        resolve_image(
            row
        )
    )


    resolved_paths.append(
        path
    )

    resolved_sources.append(
        source
    )


queue[
    '_resolved_image'
] = resolved_paths


queue[
    '_image_resolution_source'
] = resolved_sources


# ------------------------------------------------------------
# 9. Audit resolution before stopping
# ---------------------------------------------------------------

resolution_audit = (
    queue[
        [
            'review_order',
            'region_id',
            'page',
            '_resolved_image',
            '_image_resolution_source',
        ]
    ]
    .copy()
)


resolution_audit[
    '_resolved_image'
] = (
    resolution_audit[
        '_resolved_image'
    ]
    .map(
        lambda value:
            str(
                value
            )
            if value is not None
            else ''
    )
)


resolution_audit.to_csv(
    AUDIT
    / 'review_image_resolution.csv',
    index=False,
)


missing = (
    queue[
        queue[
            '_resolved_image'
        ].isna()
    ][
        [
            'region_id',
            'page',
            '_queue_reason',
        ]
    ]
    .copy()
)


missing.to_csv(
    AUDIT
    / 'missing_review_images.csv',
    index=False,
)


print(
    '\nImage-resolution sources:'
)


print(
    queue[
        '_image_resolution_source'
    ].value_counts(
        dropna=False
    )
)


if len(
    missing
):

    print(
        '\nMissing pages:'
    )

    print(
        missing[
            'page'      ]
        .value_counts()
        .head(
            30
        )
        .to_string()
    )


require(
    len(
        missing
    )
    == 0,
    (
        f'{len(missing)} queue rows still have no '
        'resolvable authoritative observed page. '
        'See audit/missing_review_images.csv and '
        'audit/review_image_resolution.csv.'
    ),
)


# ------------------------------------------------------------
# 10. Validate every resolved image
# ------------------------------------------------------------

image_identity_rows = []


for page, group in queue.groupby(
    'page',
    sort=True,
):

    paths = sorted(
        {
            Path(
                path
            ).resolve()
            for path
            in group[
                '_resolved_image'
            ]
        }
    )


    require(
        len(
            paths
        )
        == 1,
        (
            f'{page}: queue rows resolved to multiple '
            f'observed pages: {paths}'
        ),
    )


    path = paths[
        0
    ]


    image = cv2.imread(
        str(
            path
        ),
        cv2.IMREAD_GRAYSCALE,
    )


    require(
        image is not None,
        (
            f'{page}: resolved observed page is '
            f'unreadable: {path}'
        ),
    )


    image_identity_rows.append({
        'page':
            page,

        'observed_path':
            str(
                path
            ),

        'height':
            int(
                image.shape[
                    0
                ]
            ),

        'width':
            int(
                image.shape[
                    1
                ]
            ),

        'sha256':
            sha256_file(
                path
            ),

        'queue_regions':
            int(
                len(
                    group
                )
            ),
    })


pd.DataFrame(
    image_identity_rows
).to_csv(
    AUDIT
    / 'resolved_observed_page_identity.csv',
    index=False,
)


queue.to_csv(
    TABLES
    / 'adjudication_queue_resolved.csv',
    index=False,
)


print(
    '\nResolved:',
    len(
        queue
    ),
    'review regions across',
    queue[
        'page'
    ].nunique(),
    'pages.'
)

In [ ]:
# ============================================================
# Render one 59R4A adjudication candidate
#
# Input:
#   row['_resolved_image']  authoritative observed page
#   row['_x0'], row['_y0'], row['_x1'], row['_y1']
#
# Output:
#   RGB crop containing substantial surrounding context
#   with the candidate region outlined in red.
#
# The function fails closed if:
#   - image is missing/unreadable;
#   - geometry is non-finite;
#   - bounding box is invalid;
#   - bounding box falls outside the resolved page.
# ============================================================


def render(
    row,
    margin=160,
):

    # --------------------------------------------------------
    # 1. Resolve and read authoritative observed page
    # --------------------------------------------------------

    raw_path = row[
        '_resolved_image'
    ]


    require(
        raw_path is not None,
        (
            f'{row.region_id}: _resolved_image is missing.'
        ),
    )


    image_path = Path(
        raw_path
    )


    require(
        image_path.is_file(),
        (
            f'{row.region_id}: resolved observed page '
            f'does not exist: {image_path}'
        ),
    )


    observed = cv2.imread(
        str(
            image_path
        ),
        cv2.IMREAD_GRAYSCALE,
    )


    require(
        observed is not None,
        (
            f'{row.region_id}: resolved observed page '
            f'is unreadable: {image_path}'
        ),
    )


    require(
        observed.ndim
        == 2,
        (
            f'{row.region_id}: expected a greyscale '
            'observed page.'
        ),
    )


    height, width = (
        observed.shape
    )


    # --------------------------------------------------------
    # 2. Read candidate geometry
    # --------------------------------------------------------

    geometry = np.asarray(
        [
            row[
                '_x0'
            ],
            row[
                '_y0'
            ],
            row[
                '_x1'
            ],
            row[
                '_y1'
            ],
        ],
        dtype=np.float64,
    )


    require(
        np.isfinite(
            geometry
        ).all(),
        (
            f'{row.region_id}: candidate geometry is '
            f'incomplete/non-finite: {geometry.tolist()}'
        ),
    )


    (
        x0,
        y0,
        x1,
        y1,
    ) = [
        int(
            round(
                value
            )
        )
        for value
        in geometry
    ]


    # --------------------------------------------------------
    # 3. Validate geometry against page
    # --------------------------------------------------------

    require(
        x1
        > x0
        and y1
        > y0,
        (
            f'{row.region_id}: invalid candidate box '
            f'({x0}, {y0}, {x1}, {y1}).'
        ),
    )


    require(
        x0
        >= 0
        and y0
        >= 0
        and x1
        <= width
        and y1
        <= height,
        (
            f'{row.region_id}: candidate box '
            f'({x0}, {y0}, {x1}, {y1}) lies outside '
            f'page shape width={width}, height={height}.'
        ),
    )


    # --------------------------------------------------------
    # 4. Context crop
    #
    # Use at least the configured margin, but increase the
    # surrounding context for larger candidates.
    # --------------------------------------------------------

    box_width = (
        x1
        - x0
    )

    box_height = (
        y1
        - y0
    )


    adaptive_margin = max(
        int(
            margin
        ),
        int(
            round(
                1.75
                * max(
                    box_width,
                    box_height,
                )
            )
        ),
    )


    crop_x0 = max(
        0,
        x0
        - adaptive_margin,
    )

    crop_y0 = max(
        0,
        y0
        - adaptive_margin,
    )

    crop_x1 = min(
        width,
        x1
        + adaptive_margin,
    )

    crop_y1 = min(
        height,
        y1
        + adaptive_margin,
    )


    require(
        crop_x1
        > crop_x0
        and crop_y1
        > crop_y0,
        (
            f'{row.region_id}: calculated review crop '
            'is empty.'
        ),
    )


    crop = observed[
        crop_y0:crop_y1,
        crop_x0:crop_x1,
    ]


    # --------------------------------------------------------
    # 5. Convert to RGB for Gradio and annotations
    # --------------------------------------------------------

    rgb = cv2.cvtColor(
        crop,
        cv2.COLOR_GRAY2RGB,
    )


    # Candidate coordinates in crop space.

    local_x0 = (
        x0
        - crop_x0
    )

    local_y0 = (
        y0
        - crop_y0
    )

    local_x1 = (
        x1
        - crop_x0
    )

    local_y1 = (
        y1
        - crop_y0
    )


    # --------------------------------------------------------
    # 6. Draw candidate rectangle
    #
    # OpenCV is drawing on an RGB array here, so (255,0,0)
    # gives a red box in the Gradio display.
    # --------------------------------------------------------

    rectangle_thickness = max(
        2,
        int(
            round(
                min(
                    rgb.shape[
                        :2
                    ]
                )
                / 260
            )
        ),
    )


    cv2.rectangle(
        rgb,
        (
            local_x0,
            local_y0,
        ),
        (
            max(
                local_x0,
                local_x1
                - 1,
            ),
            max(
                local_y0,
                local_y1
                - 1,
            ),
        ),
        (
            255,
            0,
            0,
        ),
        rectangle_thickness,
    )


    # --------------------------------------------------------
    # 7. Add a small candidate marker
    #
    # Useful when the candidate itself is only a few pixels.
    # --------------------------------------------------------

    centre_x = int(
        round(
            (
                local_x0
                + local_x1
            )
            / 2
        )
    )

    centre_y = int(
        round(
            (
                local_y0
                + local_y1
            )
            / 2
        )
    )


    marker_size = max(
        5,
        rectangle_thickness
        * 3,
    )


    # Only add the centre cross when the candidate is very
    # small. Do not obscure larger semantic regions.

    if (
        box_width
        <= 12
        or box_height
        <= 12
    ):

        cv2.line(
            rgb,
            (
                max(
                    0,
                    centre_x
                    - marker_size,
                ),
                centre_y,
            ),
            (
                min(
                    rgb.shape[
                        1
                    ]
                    - 1,
                    centre_x
                    + marker_size,
                ),
                centre_y,
            ),
            (
                255,
                0,
                0,
            ),
            max(
                1,
                rectangle_thickness
                - 1,
            ),
        )


        cv2.line(
            rgb,
            (
                centre_x,
                max(
                    0,
                    centre_y
                    - marker_size,
                ),
            ),
            (
                centre_x,
                min(
                    rgb.shape[
                        0
                    ]
                    - 1,
                    centre_y
                    + marker_size,
                ),
            ),
            (
                255,
                0,
                0,
            ),
            max(
                1,
                rectangle_thickness
                - 1,
            ),
        )


    # --------------------------------------------------------
    # 8. Add a compact information banner
    # --------------------------------------------------------

    banner_height = 58


    canvas = np.full(
        (
            rgb.shape[
                0
            ]
            + banner_height,

            rgb.shape[
                1
            ],

            3,
        ),
        255,
        dtype=np.uint8,
    )


    canvas[
        banner_height:
    ] = rgb


    page = str(
        row[
            'page'
        ]
    )


    region_id = str(
        row[
            'region_id'
        ]
    )


    # Keep labels short so they remain readable.

    line_1 = (
        f'{page} | {region_id}'
    )


    line_2 = (
        f'candidate {box_width}x{box_height} px'
    )


    font_scale = max(
        0.38,
        min(
            0.60,
            canvas.shape[
                1
            ]
            / 1250.0,
        ),
    )


    cv2.putText(
        canvas,
        line_1,
        (
            10,
            22,
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        font_scale,
        (
            25,
            25,
            25,
        ),
        1,
        cv2.LINE_AA,
    )


    cv2.putText(
        canvas,
        line_2,
        (
            10,
            46,
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        font_scale,
        (
            70,
            70,
            70,
        ),
        1,
        cv2.LINE_AA,
    )


    return canvas

In [ ]:
ADJ_PATH=TABLES/'review_adjudications.csv'
LABELS=['ANNOTATION','PRINT','MIXED','PAGE_SCAN_EDGE','UNCERTAIN','REGION_FRAGMENT','MASK_ERROR']
STATE={'position':0}

def load_existing():
    if ADJ_PATH.exists(): return pd.read_csv(ADJ_PATH,dtype=str).fillna('')
    return pd.DataFrame(columns=['region_id','adjudicated_semantic_label','adjudicated_annotation_family','reviewer','reason','adjudicated_utc','original_semantic_label','original_annotation_family','queue_reason'])

def current_row():
    return queue.iloc[min(STATE['position'],len(queue)-1)]


def save(label,reason,reviewer):
    require(label in LABELS,'Choose one adjudication label.')
    r=current_row(); old=load_existing(); now=datetime.now(timezone.utc).isoformat()
    row={'region_id':r.region_id,'adjudicated_semantic_label':label,'adjudicated_annotation_family':'NOT_APPLICABLE' if label not in ['ANNOTATION','MIXED'] else r.annotation_family,'reviewer':reviewer.strip() or 'David Bolton','reason':reason.strip(),'adjudicated_utc':now,'original_semantic_label':r.semantic_label,'original_annotation_family':r.annotation_family,'queue_reason':r._queue_reason}
    old=old[old.region_id!=r.region_id]
    pd.concat([old,pd.DataFrame([row])],ignore_index=True).to_csv(ADJ_PATH,index=False)
    STATE['position']=min(STATE['position']+1,len(queue)-1)
    return show()

def previous():
    STATE['position']=max(0,STATE['position']-1); return show()

def next_without_save():
    STATE['position']=min(len(queue)-1,STATE['position']+1); return show()

with gr.Blocks(title='Notebook 59R4A Targeted Adjudication') as app:
    gr.Markdown('# 59R4A Targeted Adjudication')
    status=gr.Textbox(label='Candidate',interactive=False)
    image=gr.Image(label='Context with candidate box',height=720)
    label=gr.Radio(LABELS,label='Correct semantic outcome')
    reason=gr.Textbox(label='Optional adjudication reason')
    reviewer=gr.Textbox(value='David Bolton',label='Reviewer')
    progress=gr.Textbox(label='Progress',interactive=False)
    with gr.Row():
        prev=gr.Button('Previous'); save_next=gr.Button('Save + next',variant='primary'); skip=gr.Button('Next without saving')
    app.load(show,outputs=[image,status,label,progress])
    save_next.click(save,[label,reason,reviewer],[image,status,label,progress])
    prev.click(previous,outputs=[image,status,label,progress])
    skip.click(next_without_save,outputs=[image,status,label,progress])
app.launch(share=PUBLIC_LINK,debug=False,show_error=True)


In [ ]:
# Run after closing/completing the reviewer.
adj=load_existing()
summary=(adj.groupby(['queue_reason','adjudicated_semantic_label'],as_index=False).size().rename(columns={'size':'regions'})) if len(adj) else pd.DataFrame()
summary.to_csv(TABLES/'adjudication_summary.csv',index=False)
record={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'59R4A','source_59R4_run':str(SOURCE),'source_predictions_sha256':sha256_file(PRED_PATH),'source_decisions_sha256':sha256_file(DECISIONS_PATH),'queue_rows':len(queue),'queue_pages':int(queue.page.nunique()),'adjudicated_rows':len(adj),'original_decisions_modified':False,'adjudication_file':str(ADJ_PATH) if ADJ_PATH.exists() else None}
atomic_json(OUT/'notebook59R4A_verdict.json',record)
(OUT/'executive_summary.txt').write_text(chr(10).join(['NOTEBOOK 59R4A TARGETED ADJUDICATION','',f'Queue: {len(queue)} regions across {queue.page.nunique()} pages',f'Adjudicated: {len(adj)}',f'Adjudication file: {ADJ_PATH}','Original Notebook 66 decisions modified: False']))

# Curated upload package.
files=[OUT/'executive_summary.txt',OUT/'notebook59R4A_verdict.json',TABLES/'adjudication_queue.csv',TABLES/'adjudication_queue_resolved.csv']
if ADJ_PATH.exists(): files.append(ADJ_PATH)
if (TABLES/'adjudication_summary.csv').exists(): files.append(TABLES/'adjudication_summary.csv')
rows=[]
for p in files:
    require(p.exists(),f'Missing upload artefact {p}')
    dst=UPLOAD/p.name; shutil.copy2(p,dst); rows.append({'file':p.name,'sha256':sha256_file(dst),'bytes':dst.stat().st_size})
pd.DataFrame(rows).to_csv(UPLOAD/'upload_manifest.csv',index=False)
zip_path=OUT/'notebook59R4A_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.iterdir(): z.write(p,p.name)
print('59R4A complete:',OUT)
print('Adjudications:',ADJ_PATH)
print('Upload bundle:',zip_path)
